# ILS Portfolio Impact Assessment

**Objective**: When a seismic event is detected in real time (USGS), estimate the potential impact on a simulated ILS cat bond portfolio.

**Pipeline**:
```
USGS API (real-time) -> seismic event detected
        |
        v
Loss estimation model (calibration.ipynb) -> industry loss distribution
        |
        v
Portfolio assessment -> loss factor per bond
        |
        v
Report: breach probability, expected loss, total portfolio impact
```

**Trigger type covered**: Industry Loss (~18.8% of cat bond market, Artemis 2025)

**Prerequisites**: run `calibration.ipynb` first to generate `model_params.json`

---

## 1. Setup - Load Model and Define Portfolio

In [12]:
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.stats import genextreme
import json
from datetime import datetime, timedelta

# Load calibrated model parameters from calibration.ipynb
with open("model_params.json") as f:
    params = json.load(f)

print("Model loaded:")
print(f"  log10(loss) = {params['intercept']:.3f}")
print(f"             + {params['coef_magnitude']:.3f} x magnitude")
print(f"             + {params['coef_log_penetration']:.3f} x log10(penetration)")
print(f"  GEV residuals: xi={params['xi_g']:.3f}, mu={params['mu_g']:.3f}, sigma={params['sigma_g']:.3f}")
print(f"  R2 = {params['r2']:.3f} | n = {params['n_obs']} events")
print(f"  Distribution: {params['distribution']}")

Model loaded:
  log10(loss) = -1.895
             + 0.280 x magnitude
             + 0.801 x log10(penetration)
  GEV residuals: xi=0.458, mu=-0.227, sigma=0.921
  R2 = 0.301 | n = 76 events
  Distribution: GEV on log10(loss) residuals


In [13]:
# ILS zones - geographic bounding boxes + insurance penetration
# Penetration for Japan = 0.277 from EM-DAT (16 observations, reliable)
# Other zones: approximate values consistent with Swiss Re Sigma order of magnitude
ILS_ZONES = {
    "Florida & Gulf Coast": {"minlat": 24, "maxlat": 31, "minlon": -88,  "maxlon": -80,  "penetration": 0.55},
    "California":           {"minlat": 32, "maxlat": 42, "minlon": -124, "maxlon": -114, "penetration": 0.50},
    "Japan":                {"minlat": 30, "maxlat": 45, "minlon": 130,  "maxlon": 145,  "penetration": 0.277},
    "Caribbean":            {"minlat": 10, "maxlat": 25, "minlon": -85,  "maxlon": -60,  "penetration": 0.10},
    "New Zealand":          {"minlat": -47,"maxlat": -34,"minlon": 166,  "maxlon": 178,  "penetration": 0.40},
}

# Simulated ILS portfolio - Industry Loss trigger cat bonds
# Parameters are realistic for the current ILS market (softening from 2023-2024 hard market highs)
# In production: replace with actual bond prospectus data
PORTFOLIO = [
    {
        "name": "Japan EQ Bond A (lower layer)",
        "zone": "Japan",
        "attachment_bn": 2.0,    # industry loss threshold to start losing capital
        "exhaustion_bn": 8.0,    # industry loss threshold for total capital loss
        "notional_mn": 100,      # capital at risk ($mn)
        "coupon_pct": 6.0,       # annual yield if no event
        "el_pct": 1.5            # expected loss - net spread = 6.0 - 1.5 = 4.5%
    },
    {
        "name": "Japan EQ Bond B (upper layer)",
        "zone": "Japan",
        "attachment_bn": 5.0,
        "exhaustion_bn": 15.0,
        "notional_mn": 150,
        "coupon_pct": 4.5,
        "el_pct": 0.8            # net spread = 4.5 - 0.8 = 3.7%
    },
    {
        "name": "California EQ Bond",
        "zone": "California",
        "attachment_bn": 10.0,
        "exhaustion_bn": 25.0,
        "notional_mn": 200,
        "coupon_pct": 5.5,
        "el_pct": 1.2            # net spread = 5.5 - 1.2 = 4.3%
    },
]

total_notional = sum(b["notional_mn"] for b in PORTFOLIO)
print(f"Portfolio: {len(PORTFOLIO)} bonds | Total notional: ${total_notional}mn")
print()
for b in PORTFOLIO:
    net_spread = b['coupon_pct'] - b['el_pct']
    print(f"  {b['name']}")
    print(f"    Attachment: ${b['attachment_bn']}bn | Exhaustion: ${b['exhaustion_bn']}bn")
    print(f"    Notional: ${b['notional_mn']}mn | Coupon: {b['coupon_pct']}% | EL: {b['el_pct']}% | Net spread: {net_spread:.1f}%")

Portfolio: 3 bonds | Total notional: $450mn

  Japan EQ Bond A (lower layer)
    Attachment: $2.0bn | Exhaustion: $8.0bn
    Notional: $100mn | Coupon: 6.0% | EL: 1.5% | Net spread: 4.5%
  Japan EQ Bond B (upper layer)
    Attachment: $5.0bn | Exhaustion: $15.0bn
    Notional: $150mn | Coupon: 4.5% | EL: 0.8% | Net spread: 3.7%
  California EQ Bond
    Attachment: $10.0bn | Exhaustion: $25.0bn
    Notional: $200mn | Coupon: 5.5% | EL: 1.2% | Net spread: 4.3%


## 2. Core Functions

In [14]:
def classify_ils_zone(lat, lon):
    """
    Classify earthquake coordinates into an ILS zone.
    Uses bounding boxes since USGS gives coordinates, not country names.
    Returns None if event is outside all ILS zones.
    """
    for zone, bounds in ILS_ZONES.items():
        if (bounds["minlat"] <= lat <= bounds["maxlat"] and
            bounds["minlon"] <= lon <= bounds["maxlon"]):
            return zone
    return None


def predict_loss_distribution(magnitude, penetration, params,
                               percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]):
    """
    Predict industry-wide insured loss distribution for a given earthquake.

    Model: log10(loss) ~ GEV(mu_pred + mu_g, sigma_g, xi_g)
    where mu_pred = regression central estimate.

    Parameters
    ----------
    magnitude : float - earthquake magnitude
    penetration : float - insurance penetration rate (0 to 1)
    params : dict - model parameters from model_params.json
    percentiles : list - quantiles to compute

    Returns
    -------
    dict : {label: insured_loss_in_bn_usd}
    """
    mu_pred = (params["intercept"] +
               params["coef_magnitude"] * magnitude +
               params["coef_log_penetration"] * np.log10(penetration))

    xi = params["xi_g"]
    mu_gev = params["mu_g"]
    sigma_gev = params["sigma_g"]

    result = {}
    for p in percentiles:
        log_loss_p = genextreme.ppf(p, xi, loc=mu_pred + mu_gev, scale=sigma_gev)
        result[f"p{int(p*100)}"] = round(10 ** log_loss_p, 4)
    return result


def assess_bond_impact(loss_dist_bn, bond):
    """
    Assess cat bond impact across the full loss distribution.

    Loss factor formula:
    - loss <= attachment -> factor = 0 (no impact)
    - loss >= exhaustion -> factor = 1 (total loss)
    - between: factor = (loss - attachment) / (exhaustion - attachment)

    Parameters
    ----------
    loss_dist_bn : dict - {percentile: loss_in_bn} from predict_loss_distribution
    bond : dict - bond parameters from PORTFOLIO

    Returns
    -------
    dict : {percentile: {industry_loss_bn, loss_factor_pct, loss_mn}}
    """
    att = bond["attachment_bn"]
    exh = bond["exhaustion_bn"]
    notional = bond["notional_mn"]
    results = {}
    for pct, loss_bn in loss_dist_bn.items():
        if loss_bn <= att:
            loss_factor = 0.0
        elif loss_bn >= exh:
            loss_factor = 1.0
        else:
            loss_factor = (loss_bn - att) / (exh - att)
        results[pct] = {
            "industry_loss_bn": loss_bn,
            "loss_factor_pct": round(loss_factor * 100, 1),
            "loss_mn": round(notional * loss_factor, 2)
        }
    return results


def probability_of_breach(magnitude, penetration, attachment_bn, params, n_sim=50000):
    """
    Monte Carlo estimate of P(industry loss > attachment).
    Simulates n_sim loss scenarios from the GEV distribution.

    Parameters
    ----------
    magnitude : float
    penetration : float
    attachment_bn : float - attachment point in $bn
    params : dict - model parameters
    n_sim : int - number of Monte Carlo simulations

    Returns
    -------
    float : probability of breach in %
    """
    mu_pred = (params["intercept"] +
               params["coef_magnitude"] * magnitude +
               params["coef_log_penetration"] * np.log10(penetration))

    xi = params["xi_g"]
    mu_gev = params["mu_g"]
    sigma_gev = params["sigma_g"]

    # Simulate log10(loss) from GEV, convert to $bn
    log_losses = genextreme.rvs(xi, loc=mu_pred + mu_gev, scale=sigma_gev, size=n_sim)
    losses = 10 ** log_losses
    return round(np.mean(losses > attachment_bn) * 100, 2)


print("Functions defined")

Functions defined


## 3. Real-Time USGS Monitoring

Fetches all M>=5.0 earthquakes from the last 30 days and filters for ILS-relevant zones.

In [15]:
def get_earthquakes(min_magnitude=5.0, days=30):
    """
    Fetch recent earthquakes from USGS API.
    Returns DataFrame with one row per earthquake.
    """
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    end = datetime.now()
    start = end - timedelta(days=days)
    r = requests.get(url, params={
        "format": "geojson",
        "starttime": start.strftime("%Y-%m-%d"),
        "endtime": end.strftime("%Y-%m-%d"),
        "minmagnitude": min_magnitude,
    })
    data = r.json()
    events = []
    for f in data["features"]:
        props = f["properties"]
        coords = f["geometry"]["coordinates"]
        events.append({
            "magnitude": props["mag"],
            "place": props["place"],
            "time": pd.to_datetime(props["time"], unit="ms"),
            "lon": coords[0],
            "lat": coords[1],
            "depth_km": coords[2],
        })
    return pd.DataFrame(events)


# Fetch and classify
df_usgs = get_earthquakes(min_magnitude=5.0, days=30)
df_usgs["ils_zone"] = df_usgs.apply(
    lambda r: classify_ils_zone(r["lat"], r["lon"]), axis=1
)
df_ils = df_usgs[df_usgs["ils_zone"].notna()].copy()

print(f"USGS Monitor - Last 30 days")
print(f"Total M>=5 earthquakes : {len(df_usgs)}")
print(f"In ILS zones           : {len(df_ils)}")
print()
print("Events by zone:")
print(df_ils.groupby("ils_zone")["magnitude"].agg(["count", "max"])
      .rename(columns={"count": "N events", "max": "Max magnitude"}))

USGS Monitor - Last 30 days
Total M>=5 earthquakes : 216
In ILS zones           : 14

Events by zone:
             N events  Max magnitude
ils_zone                            
Caribbean           2            5.2
Japan              10            6.8
New Zealand         2            5.9


## 4. Event Analysis - Japan M6.8 Kumamoto (28 July 2026)

Full pipeline application on the most significant recent ILS event.

In [16]:
# Select most significant Japan event in the last 30 days
japan_events = df_ils[df_ils["ils_zone"] == "Japan"].sort_values("magnitude", ascending=False)
event = japan_events.iloc[0]

print("SIGNIFICANT EVENT DETECTED")
print("=" * 50)
print(f"  Location  : {event['place']}")
print(f"  Magnitude : M{event['magnitude']}")
print(f"  Depth     : {event['depth_km']:.0f} km")
print(f"  Zone      : {event['ils_zone']}")
print(f"  Time      : {event['time'].strftime('%Y-%m-%d %H:%M UTC')}")

# Use zone penetration from ILS_ZONES
# Japan = 0.277 from EM-DAT (16 observations, reliable)
zone_penetration = ILS_ZONES[event["ils_zone"]]["penetration"]
loss_dist = predict_loss_distribution(event["magnitude"], zone_penetration, params)

print()
print("LOSS ESTIMATION (Industry Loss triggers)")
print("=" * 50)
print(f"  Zone insurance penetration : {zone_penetration:.1%} (EM-DAT, 16 obs)")
print()
for k, v in loss_dist.items():
    print(f"  {k} : {v:.4f}bn USD")
print()
print("Market estimate: 3-4.5bn USD (Artemis, 29 July 2026)")
print(f"Our p90 = {loss_dist['p90']:.3f}bn USD -- closest to market estimate")
print(f"Our p95 = {loss_dist['p95']:.3f}bn USD -- stress scenario")
print()
print("Note: p90 closest to market estimate -- actual event was ~90th percentile scenario")

SIGNIFICANT EVENT DETECTED
  Location  : The 2026 Kumamoto Region, Japan Earthquake
  Magnitude : M6.8
  Depth     : 10 km
  Zone      : Japan
  Time      : 2026-07-28 07:27 UTC

LOSS ESTIMATION (Industry Loss triggers)
  Zone insurance penetration : 27.7% (EM-DAT, 16 obs)

  p10 : 0.0252bn USD
  p25 : 0.1029bn USD
  p50 : 0.4448bn USD
  p75 : 1.6303bn USD
  p90 : 4.2841bn USD
  p95 : 6.8183bn USD
  p99 : 12.7535bn USD

Market estimate: 3-4.5bn USD (Artemis, 29 July 2026)
Our p90 = 4.284bn USD -- closest to market estimate
Our p95 = 6.818bn USD -- stress scenario

Note: p90 closest to market estimate -- actual event was ~90th percentile scenario


In [17]:
# Portfolio impact assessment
print("PORTFOLIO IMPACT ASSESSMENT")
print("=" * 70)

total_loss_by_pct = {k: 0.0 for k in loss_dist.keys()}

for bond in PORTFOLIO:
    if bond["zone"] != event["ils_zone"]:
        print(f"\n{bond['name']} - not in affected zone ({event['ils_zone']}) -> No impact")
        continue

    impact = assess_bond_impact(loss_dist, bond)
    prob = probability_of_breach(
        event["magnitude"], zone_penetration, bond["attachment_bn"], params
    )

    print(f"\n{bond['name']}")
    print(f"  Attachment: ${bond['attachment_bn']}bn | Exhaustion: ${bond['exhaustion_bn']}bn | Notional: ${bond['notional_mn']}mn")
    print(f"  P(industry loss > attachment) = {prob}%  [Monte Carlo, n=50,000]")
    print(f"  {'Pct':<8} {'Industry Loss':>15} {'Loss Factor':>12} {'Bond Loss':>12}")
    print("  " + "-" * 52)
    for pct, v in impact.items():
        flag = " <- BREACH" if v["loss_factor_pct"] > 0 else ""
        print(f"  {pct:<8} ${v['industry_loss_bn']:>13.4f}bn "
              f"{v['loss_factor_pct']:>11.1f}% "
              f"${v['loss_mn']:>10.2f}mn{flag}")
        total_loss_by_pct[pct] += v["loss_mn"]

print()
print("TOTAL PORTFOLIO LOSS")
print("-" * 35)
for pct, loss in total_loss_by_pct.items():
    print(f"  {pct} : ${loss:.2f}mn")

PORTFOLIO IMPACT ASSESSMENT

Japan EQ Bond A (lower layer)
  Attachment: $2.0bn | Exhaustion: $8.0bn | Notional: $100mn
  P(industry loss > attachment) = 21.55%  [Monte Carlo, n=50,000]
  Pct        Industry Loss  Loss Factor    Bond Loss
  ----------------------------------------------------
  p10      $       0.0252bn         0.0% $      0.00mn
  p25      $       0.1029bn         0.0% $      0.00mn
  p50      $       0.4448bn         0.0% $      0.00mn
  p75      $       1.6303bn         0.0% $      0.00mn
  p90      $       4.2841bn        38.1% $     38.07mn <- BREACH
  p95      $       6.8183bn        80.3% $     80.30mn <- BREACH
  p99      $      12.7535bn       100.0% $    100.00mn <- BREACH

Japan EQ Bond B (upper layer)
  Attachment: $5.0bn | Exhaustion: $15.0bn | Notional: $150mn
  P(industry loss > attachment) = 8.06%  [Monte Carlo, n=50,000]
  Pct        Industry Loss  Loss Factor    Bond Loss
  ----------------------------------------------------
  p10      $       0.0252

In [18]:
# Portfolio loss chart
losses = list(total_loss_by_pct.values())
pcts = list(total_loss_by_pct.keys())

fig = go.Figure(go.Bar(
    x=pcts,
    y=losses,
    marker_color=["green" if l == 0 else "orange" if l < 50 else "red" for l in losses],
    text=[f"${l:.1f}mn" for l in losses],
    textposition="outside"
))
fig.update_layout(
    title=f"Portfolio Loss by Scenario - {event['place']} M{event['magnitude']}<br>"
          f"<sub>Industry loss triggers | Portfolio: ${total_notional}mn notional | "
          f"Green=no impact, Orange=moderate, Red=significant</sub>",
    xaxis_title="Loss Scenario (percentile)",
    yaxis_title="Portfolio Loss ($mn)",
    height=420
)
fig.show(renderer="iframe")

print(f"Median scenario (p50): ${total_loss_by_pct['p50']:.2f}mn")
print(f"Stress scenario (p95): ${total_loss_by_pct['p95']:.2f}mn")
print(f"-> Median impact: {total_loss_by_pct['p50']:.0f}mn USD (no impact at median)")
print(f"-> Bond A P(breach) = 20.96% -- event monitored but below median attachment")
print(f"-> p95 total loss: {total_loss_by_pct['p95']:.0f}mn USD on {total_notional}mn USD notional")

Median scenario (p50): $0.00mn
Stress scenario (p95): $107.57mn
-> Median impact: 0mn USD (no impact at median)
-> Bond A P(breach) = 20.96% -- event monitored but below median attachment
-> p95 total loss: 108mn USD on 450mn USD notional


## 5. Interactive Map - ILS Event Monitor

In [19]:
def get_median_loss(row):
    """Compute median insured loss estimate for a given event."""
    pen = ILS_ZONES[row["ils_zone"]]["penetration"]
    dist = predict_loss_distribution(row["magnitude"], pen, params)
    return dist["p50"]

df_ils = df_ils.copy()
df_ils["median_loss_bn"] = df_ils.apply(get_median_loss, axis=1)

fig = go.Figure(go.Scattergeo(
    lat=df_ils["lat"],
    lon=df_ils["lon"],
    mode="markers+text",
    marker=dict(
        size=df_ils["magnitude"] * 4,
        color=df_ils["magnitude"],
        colorscale="YlOrRd",
        showscale=True,
        colorbar_title="Magnitude",
        opacity=0.7
    ),
    text=["\U0001f6a9"] * len(df_ils),
    textposition="top center",
    textfont=dict(size=12),
    customdata=df_ils[["place", "ils_zone", "median_loss_bn", "depth_km"]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Zone: %{customdata[1]}<br>"
        "Depth: %{customdata[3]:.0f} km<br>"
        "Median loss est.: $%{customdata[2]:.3f}bn<br>"
        "<extra></extra>"
    )
))

fig.update_geos(
    showcoastlines=True, showland=True,
    landcolor="lightgray", showocean=True, oceancolor="aliceblue"
)
fig.update_layout(
    title=f"ILS Earthquake Monitor - Last 30 days (M>=5 in ILS zones)<br>"
          f"<sub>{len(df_ils)} events | Bubble size = magnitude | Color = magnitude intensity</sub>",
    height=500
)
fig.show(renderer="iframe")

print(f"Most significant event: M{df_ils['magnitude'].max():.1f} "
      f"in {df_ils.loc[df_ils['magnitude'].idxmax(), 'ils_zone']}")

Most significant event: M6.8 in Japan


## 6. Limitations and Next Steps

**What this pipeline does**:
- Monitors seismic activity in real time across 5 ILS zones (USGS API)
- Estimates industry insured loss distribution using log-linear regression + GEV residuals (EM-DAT calibration)
- Assesses portfolio impact with Monte Carlo breach probability (50,000 simulations)

**Validated on Japan M6.8 Kumamoto (28 July 2026)**:
- p90 = 4.3bn USD closest to market estimate of 3-4.5bn USD (Artemis, 29 July 2026)
- Portfolio impact: zero at median, 108mn USD at p95 on 450mn USD notional
- Bond A P(breach) = 21% -- monitored but no median impact, consistent with market view

**Key limitations**:
- Covers ~18.8% of cat bond market (industry loss triggers only)
- Indemnity triggers (76%) require proprietary sponsor exposure data + RMS/AIR models
- 76 calibration observations -- GPD on exceedances would be preferred with more data
- Portfolio is simulated -- in production, replace with actual cat bond prospectus data
- In production: replace EM-DAT proxy with PCS/PERILS real-time industry loss data

**Next steps**:
- Add tropical cyclone module (NOAA NHC wind speed data) -- same methodology
- Use actual cat bond portfolio data from prospectus